In [15]:
import json
import os

from anthropic import Anthropic
from dotenv import load_dotenv
from gitsource import chunk_documents, GithubRepositoryDataReader
from minsearch import Index

from rag_helper import RAGBase

In [2]:
load_dotenv()
client = Anthropic()

In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Q1. How many lesson pages

In [4]:
len(documents)

72

### Question: How many lesson pages are in the dataset?
### Answer: 72

## Q2. Indexing and searching

In [ ]:
print(json.dumps(documents[0], indent=2))

{
  "content": "# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we'll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type \"how are\" in WhatsApp, it suggests\n\"you\" as the next word. \"How are you\" is the most common continuation.\nYour phone use

In [ ]:
def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"],
    )
    index.fit(documents)
    return index

In [6]:
index = build_index(documents)
results = index.search("How does the agentic loop keep calling the model until it stops")
results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

### Question: What's the filename of the first result?
### Answer: 01-agentic-rag/lessons/14-agentic-loop.md

## Q3. RAG

In [10]:
message = client.messages.create(
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Hello, Claude",
        }
    ],
    model="claude-haiku-4-5",
)

In [11]:
message

Message(id='msg_015kvGjtmShoMrmzpyfUsQEA', container=None, content=[TextBlock(citations=None, text="Hello! It's nice to meet you. How can I help you today?", type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=10, output_tokens=19, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [ ]:
print(f"message text: {message.content[0].text}")
print(f"stop reason: {message.stop_reason}")
print(f"input tokens: {message.usage.input_tokens}")
print(f"output tokens: {message.usage.output_tokens}")

message text: Hello! It's nice to meet you. How can I help you today?
stop reason: end_turn
input tokens: 10
output tokens: 19


In [7]:
assistant = RAGBase(
    index=index,
    llm_client=client
)

In [8]:
question = "How does the agentic loop keep calling the model until it stops?"

answer = assistant.rag(question)

In [9]:
print(answer)

{'answer': '# How the Agentic Loop Keeps Calling the Model Until It Stops\n\nThe agentic loop uses a **`while True` loop with a flag-based exit condition** to keep calling the model until it stops requesting tool calls.\n\n## The Core Mechanism\n\nThe loop works like this:\n\n```python\nwhile True:\n    # Call the model\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=messages,\n        tools=[search_tool]\n    )\n    \n    # Add response to message history\n    messages.extend(response.output)\n    \n    has_function_calls = False\n    \n    # Process the response\n    for item in response.output:\n        if item.type == "function_call":\n            # Execute the tool and add result to messages\n            call_output = make_call(item)\n            messages.append(call_output)\n            has_function_calls = True\n        elif item.type == "message":\n            # This is the final answer\n            print(item.content[0].text)\n    

In [10]:
print(f"input tokens: {answer['input_tokens']}")

input tokens: 8203


### Question: How many input (prompt) tokens did we send to the model for this request?
### Answer: 7000 (8203)

## Q4. Chunking

In [12]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [13]:
len(chunks)

295

### Question: How many chunks do you get?
### Answer: 295

## Q5. RAG with chunking

In [16]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [17]:
chunk_index = build_index(chunks)

In [18]:
chunk_assistant = RAGBase(
    chunk_index,
    client
)

In [19]:
chunk_question = "How does the agentic loop keep calling the model until it stops?"

chunk_answer = chunk_assistant.rag(chunk_question)

In [20]:
chunk_answer

{'answer': '# How the Agentic Loop Keeps Calling the Model Until It Stops\n\nThe agentic loop uses a **`while True` loop with a flag-based exit condition**. Here\'s how it works:\n\n## The Core Mechanism\n\n1. **Flag Initialization**: At the start of each iteration, a `has_function_calls` flag is set to `False`\n\n2. **Model Call**: The loop calls the model with the current messages and available tools\n\n3. **Response Processing**: The loop iterates through the model\'s response looking for function calls:\n   - If a **function call** is found: the flag is set to `has_function_calls = True`, the function is executed, and the result is added back to the messages\n   - If a **message** (final answer) is found: it\'s displayed but doesn\'t set the flag\n\n4. **Exit Condition**: After processing the response, the loop checks:\n   ```python\n   if has_function_calls == False:\n       break\n   ```\n   If no function calls were made in this iteration, the loop exits\n\n## Key Points\n\n- **

In [24]:
print(f"input tokens: {chunk_answer['input_tokens']}")

input tokens: 2659


In [25]:
answer['input_tokens'] / chunk_answer['input_tokens']

3.084994358781497

### Question: Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?
### Answer 3x fewer